# Summary Report : Mixed-Signal Analyzer (Finance & NLP)

## 1. Introduction and Background

The main objective of this project is to design and deploy an end-to-end Machine Learning pipeline capable of predicting the short-term trend of a financial asset. Specifically, the system must determine, through binary classification, whether the price of the target stock (in this case, Apple APPL) will rise or fall over a 5-day horizon.

The uniqueness and added value of this project lie in the integration of data. To best mimic a trader’s analysis, the model combines two distinct sources of information :
* Quantitative signals : Traditional time-series analysis, including price history and the calculation of technical indicators.
* Qualitative signals : Natural Language Processing (NLP) applied daily to financial news to capture market psychology and “sentiment.” 

Beyond pure prediction, the architecture of this project was designed to demonstrate complete mastery of the data value chain, including :
* Data Engineering : Creation of an automated ETL pipeline (Extraction via API, Transformation, and Loading).
* Statistical Modeling : Comparing advanced Machine Learning algorithms to solve a complex classification problem.
* Software Development : Structuring code into modular Python scripts, database management (SQL), and applying best practices for version control (Git).

![Analyse Technique AAPL](../Images/Schéma_global.png)

## 2. Feature Engineering and Signal Justification

A model’s performance depends heavily on the quality of the explanatory variables it is fed. For this pipeline, we designed a hybrid dataset that captures both the intrinsic price dynamics (technical analysis) and exogenous market sentiment (sentiment analysis).

### 2.1. Quantitative Signals : Price Dynamics and Risk

* Moving Averages (SMA_20 and EMA_20) : The 20-day simple moving average (SMA) provides a baseline for the short- to medium-term trend. The exponential moving average (EMA), by placing greater weight on recent prices, allows the model to detect trend breaks more quickly.
* Historical Volatility (14 days) : This indicator measures market uncertainty. It is calculated using the moving standard deviation of daily returns, annualized according to the formula: $Volatility = \sigma_{14} \times \sqrt{252}$. Volatility spikes often precede major market reversals, a crucial signal for decision trees.
* Relative Strength Index (RSI_14) : This is a momentum oscillator that measures the speed of price movements to identify overbought or oversold conditions. Its mathematical formula is: $RSI = 100 - \frac{100}{1 + RS}$, where $RS$ (Relative Strength) represents the ratio of the average gains to the average losses over 14 days.

### 2.2. Qualitative Signals : NLP and Market Psychology

Pure quantitative models suffer from an inherent lag because they respond only to past prices. The integration of textual data aims to give the model the ability to anticipate based on the flow of information.

* The Choice of FinBERT : General-purpose NLP models often fail to capture the nuances of financial jargon (for example, the word “drop” can be positive when referring to the unemployment rate). We implemented FinBERT, a Transformer-based model specifically retrained by ProsusAI on a massive financial corpus (Financial PhraseBank).
* Daily Aggregation (daily_sentiment and new_volumes) : FinBERT’s raw predictions (probabilities for the Positive, Negative, and Neutral classes) were weighted and aggregated on a daily basis. This allows us to transform an unstructured news feed into a continuous time series that aligns perfectly with our market data in the SQL table `fact_news_sentiment`.

Below is a chart showing the performance of Apple stock, illustrating the crossover of moving averages and the overbought/oversold zones identified by the RSI

![Analyse Technique AAPL](../Images/SMA_Close_.png)

This chart shows the daily values of the “daily_sentiment” index as well as the number of news stories about Apple

**It's important to note that these values comes from an invented dataset in order to complete a demonstrative Machine Learning model** 

![Analyse Technique AAPL](../Images/daily_sentiment.png)

## 3. Model Evaluation and Performance

### 3.1. Evaluation Methodology and Temporal Validation

In financial modeling, the use of traditional cross-validation (such as random K-fold cross-validation) should be avoided, as it would result in data leakage from the future to the past.

To ensure the integrity of our tests on Apple stock, we have implemented several tools:
* A strict chronological split (80% / 20%) : The model is trained solely on the distant past and evaluated exclusively on the most recent data.
* A TimeSeriesSplit (temporal cross-validation) : Hyperparameter optimization via GridSearchCV adhered to the chronological order of the training blocks to simulate a real trading environment under actual conditions.
* A binary classification formulation : The target variable indicates whether the closing price at a 5-day horizon will be higher ($1$) or lower ($0$) than the current price.

### 3.2. Comparative Analysis: Random Forest vs. XGBoost

The two ensemble algorithms tested revealed radically different learning dynamics on our dataset

* The Behavior of the Random Forest (Conservative Model) : The Random Forest builds independent trees in parallel. Faced with the inherent noise in stock market time series, the average of the forest’s votes adopted an extreme smoothing strategy. On the test set, the model favored the majority class, demonstrating an inability to isolate weak signals of trend reversals. Although it seeks to counter overfitting, its independent structure proved too rigid to capture the nonlinearity of mixed signals (price + sentiment).
* The behavior of XGBoost (Sequential Gradient Boosting) : In contrast, XGBoost builds its trees iteratively; each new tree is specifically trained to correct the residual errors of the previous trees. This sequential approach allowed it to adapt precisely to price fluctuations and outperform, achieving an overall accuracy higher than that of Random Forest.

### 3.3.  Analysis of Advanced Metrics (ROC, Precision-Recall and Confusion Matrix)

To validate the robustness of XGBoost, the analysis goes beyond overall accuracy :
* ROC and Precision-Recall Curves : Analysis of probability scores using the area under the curve (AUC) confirms the classifier’s robustness, demonstrating sufficient discriminatory power to support the automation of decision orders.
* The Confusion Matrix : It highlights the model’s ability to correctly identify uptrend and downtrend zones, minimizing false signals compared to the Random Forest.

**Since the XGBoost model offers better accuracy, I will only show its graphs here.**

![Analyse Technique AAPL](../Images/Confusion_matrix_xgb.png)

The confusion matrix visualizes the XGBoost model's classification performance on the unseen test dataset, comparing algorithmic predictions against actual market movements.  The distribution of the results highlights the following operational characteristics :

* Strong Upside Detection (High Recall): The model excels at identifying upward trends, correctly classifying 8 out of 9 actual market rallies (True Positives). With only 1 False Negative (predicting a drop when the market actually rose), the algorithm demonstrates a strong capacity to avoid missing profitable upside opportunities.
* Moderate Downside Precision: Performance is more balanced during market downturns. The model correctly identified 5 price drops (True Negatives) but generated 5 False Positives (predicting a rise when the market actually fell). This indicates a slightly optimistic bias in the algorithm during turbulent periods.

By minimizing False Negatives, the model proves highly relevant for a "long-biased" trading strategy. It captures the vast majority of positive market shifts while maintaining a satisfactory overall accuracy, confirming the predictive value of combining price action with NLP sentiment.

![Analyse Technique AAPL](../Images/ROC_P-R_curves_xgb.png)

ROC Curve (AUC = 0.84) : The Receiver Operating Characteristic curve demonstrates strong discriminatory capacity. With an Area Under the Curve of 0.84, the algorithm effectively distinguishes between positive (price surge) and negative (price drop) market movements, performing significantly better than a random baseline.

Precision-Recall Curve (AUC = 0.78) : In financial forecasting, false positives (predicting a breakout that fails) are particularly costly. The Precision-Recall curve confirms that the model maintains high precision even as recall increases. An AUC of 0.78 proves the model is well-calibrated, ensuring that when it triggers a "Buy" signal, the probability of an actual market rise is statistically reliable.

## 4. Feature Importance

To interpret the XGBoost model's decision-making process, we extracted the Feature Importance metric (measured by Information Gain). This allows us to quantify the exact contribution of each variable in reducing prediction uncertainty and identifying the strongest market signals.

![Analyse Technique AAPL](../Images/Feature_importance_xgb.png)

The XGBoost Feature Importance chart highlights the hierarchical weight of the variables driving the model's predictions. The distribution reveals three key insights :

* Dominance of Trend and Risk Indicators : Quantitative technical indicators are the primary drivers of the model. The 20-day Exponential and Simple Moving Averages (EMA_20, SMA_20), followed by Volatility_14, hold the highest relative importance, confirming that structural price momentum remains the foundation of the predictive engine.
* Validation of the NLP Integration : The daily_sentiment feature proves its added value by ranking solidly in the middle of the distribution. It successfully outperforms traditional market metrics such as the RSI, opening prices, and daily returns. This confirms the initial hypothesis, qualitative news sentiment actively contributes to refining the algorithm's decisions.  
* Low-Impact Variables : Features at the very bottom of the chart, notably news_volume, Daily_Return, and Open price, yield near-zero information gain. These variables are largely ignored by the decision trees and could potentially be pruned in future pipeline optimizations to reduce noise.


## Conclusion

This project validates the architecture of a fully automated, hybrid machine learning pipeline designed to predict short-term financial trends over a 5-day horizon. By seamlessly orchestrating data extraction, NLP transformations, SQL relational storage, and predictive modeling, we have built a robust, end-to-end analytical engine capable of processing mixed signals.

The core objective was to determine if natural language processing applied to financial news could improve predictions compared to a strictly price-based model. The empirical results, driven by the XGBoost algorithm, provide that it does. 
By integrating the FinBERT model to extract daily sentiment polarity, the pipeline successfully captured market psychology, allowing the algorithm to contextualize mathematical price movements and significantly reduce false positive signals during volatile periods.

### Personnal Conclusion

I undertook this project with the goal of improving my personal skills and my knowledge of the world of finance. It is far from perfect and undoubtedly quite naive in many respects, but I consider it a good practice tool and an excellent way to demonstrate my determination and my desire to learn and apply myself to concrete topics.